In [1]:
# Configuration
TARGET_COLUMN = 'AAPL'  # Must match the ticker/filename from the download notebook

# Step 1: Install only what's missing — don't force-reinstall TensorFlow/NumPy/
# pandas, since Colab's preinstalled versions are already mutually compatible
# with its current Python version. Pinning exact old versions breaks the
# moment Colab's base image moves past them (as just happened with tf==2.16.1).
print("\nStep 1: Checking and installing dependencies...")

import sys
import subprocess

def install_if_missing(import_name, pip_name=None):
    try:
        __import__(import_name)
        print(f"  {import_name}: already available")
    except ImportError:
        pip_name = pip_name or import_name
        print(f"  Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("optuna")
install_if_missing("lightgbm")
install_if_missing("xgboost")
install_if_missing("pywt", "PyWavelets")
install_if_missing("pykalman")
install_if_missing("transformers")
install_if_missing("bs4", "beautifulsoup4")
# tensorflow, torch, numpy, pandas, requests, scikit-learn are already
# preinstalled in Colab and compatible with each other — leave them alone.

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')
import time
import random
import torch

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nSetup complete. TARGET_COLUMN =", TARGET_COLUMN)


Step 1: Checking and installing dependencies...
  Installing optuna...
  lightgbm: already available
  xgboost: already available
  pywt: already available
  Installing pykalman...
  transformers: already available
  bs4: already available

Setup complete. TARGET_COLUMN = AAPL


In [2]:
print("\nStep 2: Importing libraries...")

# Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("Optuna version:", optuna.__version__)
except Exception as e:
    print(f"Optuna unavailable: {e}")
    OPTUNA_AVAILABLE = False

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (
        Dense, LSTM, Bidirectional, Conv1D, GRU, Dropout,
        MultiHeadAttention, GlobalAveragePooling1D, Input, Flatten
    )
    from tensorflow.keras.regularizers import l2
    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception as e:
    print(f"TensorFlow unavailable: {e}")
    TF_AVAILABLE = False

# PyWavelets
try:
    import pywt
    PYWT_AVAILABLE = True
except Exception as e:
    print(f"PyWavelets unavailable: {e}")
    PYWT_AVAILABLE = False

# PyKalman
try:
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True
except Exception as e:
    print(f"PyKalman unavailable: {e}")
    PYKALMAN_AVAILABLE = False

# LightGBM
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception as e:
    print(f"LightGBM unavailable: {e}")
    LGB_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception as e:
    print(f"XGBoost unavailable: {e}")
    XGB_AVAILABLE = False

# Transformers + PyTorch (for FinBERT sentiment later)
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True
    print("Transformers version:", transformers.__version__)
    print("PyTorch version:", torch.__version__)
except Exception as e:
    print(f"Transformers/PyTorch unavailable: {e}")
    TRANSFORMERS_AVAILABLE = False

# Requests + BeautifulSoup (for FinViz news scraping later)
import requests
from bs4 import BeautifulSoup

print("\nAvailability summary:")
print(f"  OPTUNA_AVAILABLE={OPTUNA_AVAILABLE}, TF_AVAILABLE={TF_AVAILABLE}, "
      f"PYWT_AVAILABLE={PYWT_AVAILABLE}, PYKALMAN_AVAILABLE={PYKALMAN_AVAILABLE}, "
      f"LGB_AVAILABLE={LGB_AVAILABLE}, XGB_AVAILABLE={XGB_AVAILABLE}, "
      f"TRANSFORMERS_AVAILABLE={TRANSFORMERS_AVAILABLE}")


Step 2: Importing libraries...
Optuna version: 5.0.0
TensorFlow version: 2.20.0
Transformers version: 5.16.1
PyTorch version: 2.11.0+cu128

Availability summary:
  OPTUNA_AVAILABLE=True, TF_AVAILABLE=True, PYWT_AVAILABLE=True, PYKALMAN_AVAILABLE=True, LGB_AVAILABLE=True, XGB_AVAILABLE=True, TRANSFORMERS_AVAILABLE=True


In [3]:
# Sanity check — confirm the runtime actually has a GPU and both
# frameworks can see it, before we run anything expensive.
import tensorflow as tf
print("TensorFlow GPU devices:", tf.config.list_physical_devices('GPU'))

print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("PyTorch GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch sees no GPU — FinBERT will run on CPU (slower, not broken).")

TensorFlow GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch CUDA available: True
PyTorch GPU: Tesla T4


In [6]:
# Step 2: Load and Clean Data
print("\nStep 2: Loading and cleaning data...")

from google.colab import files

print(f"Upload your {TARGET_COLUMN} price CSV (any filename is fine):")
uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError(f"Expected exactly 1 file, got {len(uploaded)}: {list(uploaded.keys())}")

file_name = list(uploaded.keys())[0]
print(f"Using uploaded file: {file_name}")

df = pd.read_csv(file_name)
print(f"Raw columns: {df.columns.tolist()}")
print(f"Raw rows: {len(df)}")

expected_cols = {'Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'}
missing = expected_cols - set(df.columns)
if missing:
    raise RuntimeError(f"Missing expected columns: {missing}. Got: {df.columns.tolist()}. This usually means the CSV still has the old MultiIndex header — re-check the download notebook's flattening step.")

price_col = 'Close'

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
for col in ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df[price_col] = df[price_col].interpolate().ffill().bfill()
df = df.dropna(subset=['Timestamp', price_col]).reset_index(drop=True)

stock_data = df
print(f"\nCleaned rows: {len(stock_data)}")
print(stock_data[['Timestamp', price_col]].head())
print(stock_data[['Timestamp', price_col]].tail())


Step 2: Loading and cleaning data...
Upload your AAPL price CSV (any filename is fine):


Saving AAPL (2).csv to AAPL (2) (1).csv
Using uploaded file: AAPL (2) (1).csv
Raw columns: ['Timestamp', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
Raw rows: 1950

Cleaned rows: 1950
                  Timestamp       Close
0 2026-09-14 13:30:00+00:00  333.170013
1 2026-09-14 13:31:00+00:00  332.880005
2 2026-09-14 13:32:00+00:00  332.420013
3 2026-09-14 13:33:00+00:00  332.420013
4 2026-09-14 13:34:00+00:00  332.756714
                     Timestamp       Close
1945 2026-09-18 19:55:00+00:00  336.239899
1946 2026-09-18 19:56:00+00:00  336.035004
1947 2026-09-18 19:57:00+00:00  336.179993
1948 2026-09-18 19:58:00+00:00  336.049988
1949 2026-09-18 19:59:00+00:00  335.589996


In [10]:
# Step 2.5b: Fetch full article text
if news_data is not None and not news_data.empty:
    def fetch_article_content(url, headline):
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        try:
            response = requests.get(url, headers=headers, timeout=5)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')
            article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
            return article_text if len(article_text.strip()) > 50 else headline
        except Exception:
            return headline

    print(f"Extracting article text for {len(news_data)} headlines (this may take a minute)...")
    news_data['News_detail'] = news_data.apply(
        lambda row: fetch_article_content(row['Headline_Link'], row['Headline']), axis=1
    )
    print("News gathering complete!")

print(news_data[['Ticker', 'News_Timestamp', 'Headline']].head())

Extracting article text for 50 headlines (this may take a minute)...
News gathering complete!
  Ticker      News_Timestamp  \
0   AAPL 2026-09-18 17:01:00   
1   AAPL 2026-09-18 16:21:00   
2   AAPL 2026-09-18 15:49:00   
3   AAPL 2026-09-18 15:24:00   
4   AAPL 2026-09-18 15:10:00   

                                            Headline  
0  Coinbase Files to List Single-Stock Perps on A...  
1  Apple Stock Gets Stunning Price Target Hike on...  
2                ETF tracking Nasdaq-100 outperforms  
3  Where AAPL Stock's $510 Billion Of Shareholder...  
4  Intel vs. SK Hynix: Which Chip Stock Is Worth ...  


In [12]:
# Step 2.6: Process News Data with FinBERT
if TRANSFORMERS_AVAILABLE and news_data is not None and not news_data.empty:
    print("\nStep 2.6: Processing news data with FinBERT...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    try:
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        model.to(device)
        print(f"FinBERT model loaded successfully on {device}.")
    except Exception as e:
        print(f"Failed to load FinBERT model: {e}. Proceeding without sentiment analysis.")
        TRANSFORMERS_AVAILABLE = False

    if TRANSFORMERS_AVAILABLE:
        news_data['Text_To_Analyze'] = news_data['News_detail'].fillna(news_data['Headline'])

        def get_finbert_sentiment_batch(texts):
            results = []
            batch_size = 16
            for i in range(0, len(texts), batch_size):
                batch_texts = [str(t)[:512] for t in texts[i:i+batch_size]]
                try:
                    inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                    with torch.no_grad():
                        outputs = model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    results.extend(probs.cpu().numpy())
                except Exception:
                    results.extend([[0.33, 0.33, 0.33]] * len(batch_texts))
            return np.array(results)

        print(f"Analyzing sentiment for {len(news_data)} news items...")
        sentiment_scores = get_finbert_sentiment_batch(news_data['Text_To_Analyze'].tolist())

        # FinBERT outputs logits in order: [Positive, Negative, Neutral]
        news_data[['positive', 'negative', 'neutral']] = sentiment_scores
        news_data['polarity'] = news_data['positive'] - news_data['negative']
        news_data['strength'] = news_data[['positive', 'negative', 'neutral']].max(axis=1)
        news_data['weight'] = 1.0  # simple uniform weighting — replaces the original's News_Index-based weight, which just measured scrape order and wasn't meaningful
        news_data['impact'] = news_data['polarity'] * news_data['strength'] * news_data['weight']

        print("Sentiment processing complete! Sample scores:")
        print(news_data[['Headline', 'positive', 'negative', 'neutral', 'polarity', 'impact']].head())
else:
    print("Skipping sentiment analysis — no news data or Transformers unavailable.")


Step 2.6: Processing news data with FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT model loaded successfully on cuda.
Analyzing sentiment for 50 news items...
Sentiment processing complete! Sample scores:
                                            Headline  positive  negative  \
0  Coinbase Files to List Single-Stock Perps on A...  0.029505  0.050151   
1  Apple Stock Gets Stunning Price Target Hike on...  0.205811  0.020863   
2                ETF tracking Nasdaq-100 outperforms  0.493049  0.455813   
3  Where AAPL Stock's $510 Billion Of Shareholder...  0.035406  0.201758   
4  Intel vs. SK Hynix: Which Chip Stock Is Worth ...  0.652498  0.021294   

    neutral  polarity    impact  
0  0.920344 -0.020646 -0.019001  
1  0.773326  0.184949  0.143026  
2  0.051137  0.037236  0.018359  
3  0.762835 -0.166352 -0.126899  
4  0.326208  0.631204  0.411859  


In [18]:
# Merge news impact with price data
sentiment_cols = ['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact']

if news_data is not None and not news_data.empty and not all(col in news_data.columns for col in sentiment_cols):
    print(f"WARNING: news_data is missing sentiment columns {set(sentiment_cols) - set(news_data.columns)}. "
          f"Re-run the FinBERT sentiment cell (Step 2.6) before this cell.")

try:
    if news_data is not None and not news_data.empty and all(col in news_data.columns for col in sentiment_cols):
        print("Aligning news with price data...")
        stock_df = stock_data.sort_values('Timestamp').reset_index(drop=True)

        # Drop any sentiment columns already on stock_df from a previous run
        # of this cell — otherwise merge_asof renames the collision to
        # positive_x/positive_y instead of overwriting, and the fillna
        # loop below fails looking for the plain column name.
        stock_df = stock_df.drop(columns=[c for c in sentiment_cols if c in stock_df.columns], errors='ignore')

        stock_df['Timestamp_naive'] = stock_df['Timestamp'].dt.tz_localize(None)
        news_sorted = news_data.sort_values('News_Timestamp').reset_index(drop=True)

        merged_data = pd.merge_asof(
            stock_df,
            news_sorted[['News_Timestamp'] + sentiment_cols],
            left_on='Timestamp_naive',
            right_on='News_Timestamp',
            direction='backward'
        )
        merged_data = merged_data.drop(columns=['Timestamp_naive', 'News_Timestamp'])

        for col in sentiment_cols:
            merged_data[col] = merged_data[col].fillna(0.33 if col in ['positive', 'negative', 'neutral'] else 0.0)

        stock_data = merged_data
        print(f"Merged data shape: {stock_data.shape}")
        print("Merged data sample:\n", stock_data[['Timestamp'] + sentiment_cols].head(5))
    else:
        print("No usable news/sentiment data available. Filling neutral sentiment defaults.")
        for col in sentiment_cols:
            stock_data[col] = 0.33 if col in ['positive', 'negative', 'neutral'] else 0.0

except Exception as e:
    print(f"Error merging news and price data: {e}. Proceeding without sentiment features.")
    for col in sentiment_cols:
        stock_data[col] = 0.33 if col in ['positive', 'negative', 'neutral'] else 0.0

print(f"\nRows with non-default sentiment: {(stock_data['impact'] != 0.0).sum()} / {len(stock_data)}")
print(stock_data[stock_data['Timestamp'] > '2026-09-16 16:30:00'][['Timestamp'] + sentiment_cols].head(5))

Aligning news with price data...
Merged data shape: (1950, 13)
Merged data sample:
                   Timestamp  positive  negative  neutral  polarity  strength  \
0 2026-09-14 13:30:00+00:00      0.33      0.33     0.33       0.0       0.0   
1 2026-09-14 13:31:00+00:00      0.33      0.33     0.33       0.0       0.0   
2 2026-09-14 13:32:00+00:00      0.33      0.33     0.33       0.0       0.0   
3 2026-09-14 13:33:00+00:00      0.33      0.33     0.33       0.0       0.0   
4 2026-09-14 13:34:00+00:00      0.33      0.33     0.33       0.0       0.0   

   impact  
0     0.0  
1     0.0  
2     0.0  
3     0.0  
4     0.0  

Rows with non-default sentiment: 990 / 1950
                    Timestamp  positive  negative   neutral  polarity  \
961 2026-09-16 16:31:00+00:00  0.115156  0.029143  0.855701  0.086013   
962 2026-09-16 16:32:00+00:00  0.115156  0.029143  0.855701  0.086013   
963 2026-09-16 16:33:00+00:00  0.115156  0.029143  0.855701  0.086013   
964 2026-09-16 16:34:00+00

In [16]:
print(news_data.columns.tolist())

['Ticker', 'Headline', 'Headline_Link', 'News_Timestamp_Raw', 'News_Timestamp', 'News_detail', 'Text_To_Analyze', 'positive', 'negative', 'neutral', 'polarity', 'strength', 'weight', 'impact']


In [19]:
print(stock_data['impact'].nunique(), "unique impact values across", len(stock_data), "rows")
print(stock_data[stock_data['impact'] != 0]['impact'].value_counts().head(10))

21 unique impact values across 1950 rows
impact
-0.787991    220
 0.073601    179
-0.019001    179
 0.127782     67
-0.025589     50
 0.792526     49
 0.143026     40
-0.029332     34
 0.018359     32
-0.870592     29
Name: count, dtype: int64


In [20]:
print("Distinct headline timestamps in news_data:", news_data['News_Timestamp'].nunique(), "/", len(news_data))
print("Distinct FinBERT impact scores in news_data:", news_data['impact'].nunique(), "/", len(news_data))
print(news_data[['Headline', 'News_Timestamp', 'impact']].sort_values('News_Timestamp'))

Distinct headline timestamps in news_data: 50 / 50
Distinct FinBERT impact scores in news_data: 50 / 50
                                             Headline      News_Timestamp  \
49  This Analyst Sees INTC Stock Hitting $200 In T... 2026-09-16 16:30:00   
48  S&P 500, Nasdaq, Dow Futures Inch Higher As In... 2026-09-16 19:29:00   
47  Snap targets enterprises with Salesforce, Nvid... 2026-09-16 19:45:00   
46  Going Beyond Apple's Presentation: Interesting... 2026-09-16 20:01:00   
45        Did AI Servers Really Quadruple Dell Stock? 2026-09-16 21:01:00   
44  Apple's AI ambitions are outgrowing the Mac - ... 2026-09-16 23:06:00   
43  AI risk split among tech leaders set to play o... 2026-09-16 23:36:00   
42  iPhone 18 Pro preorders rise up to 20% in Taiw... 2026-09-17 00:04:00   
41  Shaq moved his dad from a $60K Army salary to ... 2026-09-17 06:35:00   
40  How to Find and Cancel Unused Subscriptions: S... 2026-09-17 07:41:00   
39  Why Stocks Are Loving the Fed's Rate Hike Eve

In [21]:
# Confirm: how many headlines fall outside trading hours (13:30-19:59 UTC)?
news_sorted_check = news_data.sort_values('News_Timestamp').reset_index(drop=True)
news_sorted_check['time_only'] = news_sorted_check['News_Timestamp'].dt.time
in_hours = news_sorted_check['time_only'].between(pd.Timestamp('13:30').time(), pd.Timestamp('19:59').time())
print(f"Headlines within trading hours: {in_hours.sum()} / {len(news_sorted_check)}")
print(f"Headlines outside trading hours (get superseded before next price row): {(~in_hours).sum()} / {len(news_sorted_check)}")

Headlines within trading hours: 18 / 50
Headlines outside trading hours (get superseded before next price row): 32 / 50


In [23]:
# Step 3: Denoise Data for Highest SNR
print("\nStep 3: Denoising data...")
price_scaler = MinMaxScaler()
feature_columns = ['Close']
sentiment_cols = ['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact']
if all(col in stock_data.columns for col in sentiment_cols):
    feature_columns.extend(sentiment_cols)

if not stock_data.empty:
    scaled_data = price_scaler.fit_transform(stock_data[['Close']])
    print(f"Scaled data shape: {scaled_data.shape}")

    def calculate_snr(original, denoised):
        noise = original - denoised
        signal_power = np.mean(original ** 2)
        noise_power = np.var(noise)
        return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

    def optimize_denoising(trial, data):
        original_std = np.std(data)  # NEW: needed for the variance guard below
        methods = []
        if PYWT_AVAILABLE:
            wavelet = trial.suggest_categorical('wavelet', ['db1', 'db2', 'sym2'])
            level = trial.suggest_int('wavelet_level', 1, 3)
            wavelet_denoised = pywt.wavedec(data, wavelet, level=level)
            threshold = np.std(wavelet_denoised[-1]) * np.sqrt(2 * np.log(len(data)))
            wavelet_denoised = pywt.waverec([pywt.threshold(c, threshold, mode='soft') for c in wavelet_denoised], wavelet)[:len(data)]
            snr_wavelet = calculate_snr(data, wavelet_denoised)
            methods.append(('wavelet', snr_wavelet, wavelet_denoised))
        ma_window = trial.suggest_int('ma_window', 3, 10)
        ma_denoised = pd.Series(data).rolling(window=ma_window, min_periods=1).mean().values
        snr_ma = calculate_snr(data, ma_denoised)
        methods.append(('ma', snr_ma, ma_denoised))
        # REMOVED: the autoencoder branch — it built a model and called
        # .predict() without ever calling .fit(), so it wasn't actually
        # denoising anything, just returning random-weight noise dressed
        # up as a valid candidate.
        if PYKALMAN_AVAILABLE:
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=data[0])
            kalman_denoised, _ = kf.filter(data)
            snr_kalman = calculate_snr(data, kalman_denoised.flatten())
            methods.append(('kalman', snr_kalman, kalman_denoised))

        # NEW: disqualify any method that flattens the series too much,
        # before picking by SNR. This is the actual fix for the
        # zero-Sharpe/zero-volatility bug — without this line, SNR alone
        # always rewards more smoothing with nothing to stop it.
        methods = [m for m in methods if original_std == 0 or (np.std(m[2]) / original_std) >= 0.5]
        if not methods:
            fallback = pd.Series(data).rolling(window=3, min_periods=1).mean().values
            return calculate_snr(data, fallback), fallback

        best_method = max(methods, key=lambda x: x[1] if not np.isinf(x[1]) else -float('inf'))
        return best_method[1], best_method[2]

    if OPTUNA_AVAILABLE:
        print("Optimizing denoising with Optuna...")
        study_denoise = optuna.create_study(direction='maximize', sampler=TPESampler())
        study_denoise.optimize(lambda trial: optimize_denoising(trial, scaled_data[:, 0])[0], n_trials=10)
        _, denoised_data = optimize_denoising(optuna.trial.FixedTrial(study_denoise.best_params), scaled_data[:, 0])
        print(f"Best SNR after Optuna: {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")
    else:
        print("Optuna unavailable. Using default moving average denoising...")
        denoised_data = pd.Series(scaled_data[:, 0]).rolling(window=5, min_periods=1).mean().values
        print(f"Best SNR (default): {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")

    # NEW: the sanity check that would have caught the original bug immediately
    original_std = np.std(scaled_data[:, 0])
    denoised_std = np.std(denoised_data)
    print(f"Variance retained: {(denoised_std / original_std * 100) if original_std > 0 else 0:.1f}% (original std={original_std:.6f}, denoised std={denoised_std:.6f})")

    stock_denoised_data = stock_data[feature_columns].copy()
    stock_denoised_data['Denoised_Close'] = denoised_data
else:
    print("No stock data available. Exiting.")
    exit()


Step 3: Denoising data...
Scaled data shape: (1950, 1)
Optimizing denoising with Optuna...
Best SNR after Optuna: 4.20 dB
Variance retained: 99.8% (original std=0.224883, denoised std=0.224348)


In [24]:
# Step 4: Prepare Sequences
print("\nStep 4: Preparing sequences...")
SEQ_LENGTH = 30
WINDOW_SIZE = 300

# Use Denoised_Close (already scaled 0-1) instead of raw Close, so every
# feature is on the same scale — mixing a ~$330 raw price with 0-1
# sentiment features would badly distort model training.
model_feature_columns = ['Denoised_Close'] + [c for c in sentiment_cols if c in stock_denoised_data.columns]

full_data = stock_denoised_data[model_feature_columns].values
X_for_prediction = np.array([full_data[i:i + SEQ_LENGTH] for i in range(len(full_data) - SEQ_LENGTH)])
y_for_prediction = stock_denoised_data['Denoised_Close'].values[SEQ_LENGTH:]

stock_X_deep = X_for_prediction
stock_X_ml = X_for_prediction
stock_y = y_for_prediction

print(f"X_deep shape: {stock_X_deep.shape}, y shape: {stock_y.shape}")
last_300_scaled = stock_denoised_data[model_feature_columns].values[-WINDOW_SIZE:]


Step 4: Preparing sequences...
X_deep shape: (1920, 30, 7), y shape: (1920,)


In [25]:
# Step 5: Define and Optimize Models
print("\nStep 5: Defining and optimizing models...")
feature_dim = len(model_feature_columns)  # was feature_columns in the original — renamed in Step 4

def optimize_cnn_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_conv1d_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_bilstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Bidirectional(LSTM(lstm_units, kernel_regularizer=l2(0.02))),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_stacked_lstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_gru(trial):
    gru_units = trial.suggest_int('gru_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        GRU(gru_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_lightgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1.0)
    }
    return lgb.LGBMRegressor(**params, force_col_wise=True)

def optimize_xgboost(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0)
    }
    return xgb.XGBRegressor(**params)

def optimize_transformer(trial):
    num_heads = trial.suggest_int('num_heads', 4, 8)
    ff_dim = trial.suggest_int('ff_dim', 64, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(inputs, inputs)
    x = Dropout(dropout_rate)(x)
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(x, x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(ff_dim, activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_tcn(trial):
    filters = trial.suggest_int('filters', 32, 64)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = inputs
    shortcut = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(shortcut)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.Activation('relu')(x)
    x = Flatten()(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(0.02))(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

print(f"Model factories defined. feature_dim = {feature_dim}")


Step 5: Defining and optimizing models...
Model factories defined. feature_dim = 7


In [27]:
models = [
    ('cnn_lstm', optimize_cnn_lstm, 'deep'),
    ('conv1d_lstm', optimize_conv1d_lstm, 'deep'),
    ('bilstm', optimize_bilstm, 'deep'),
    ('stacked_lstm', optimize_stacked_lstm, 'deep'),
    ('gru', optimize_gru, 'deep'),
    ('lightgbm', optimize_lightgbm, 'ml'),
    ('xgboost', optimize_xgboost, 'ml'),
    ('transformer', optimize_transformer, 'deep'),
    ('tcn', optimize_tcn, 'deep')
]

# Step 6: Train and Evaluate Models with Walk-Forward Validation
print("\nStep 6: Training and evaluating models...")
def walk_forward_validation(model, model_name, X, y, price_scaler, step_size=300, max_steps=20):
    best_adj_r2 = -float('inf')
    best_model = None
    adj_r2_history = []
    n_steps = min(max_steps, max(1, (len(X) - SEQ_LENGTH) // step_size))
    if n_steps < 1:
        print(f"Error: Insufficient data for walk-forward validation for {model_name}.")
        return {'adj_r2': 0, 'best_model': model}, adj_r2_history
    n_predictors = feature_dim
    for i in range(1, n_steps + 1):
        train_end = i * step_size
        test_size = min(step_size, len(X) - train_end)
        train_X = X[:train_end]
        train_y = y[:train_end]
        test_X = X[train_end:train_end + test_size]
        test_y = y[train_end:train_end + test_size]
        if len(test_X) == 0 or len(test_y) == 0 or len(test_X) != len(test_y):
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                # FIX: was train_X[:, :, 0].reshape(-1, SEQ_LENGTH) — that
                # only kept feature 0 (price), silently dropping all 6
                # sentiment columns. Flatten all features instead.
                model.fit(train_X.reshape(train_X.shape[0], -1), train_y)
                preds = model.predict(test_X.reshape(test_X.shape[0], -1))
            else:
                model.fit(train_X, train_y, epochs=10, batch_size=64, validation_split=0.2, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
                preds = model.predict(test_X, verbose=0).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
            ss_res = np.sum((test_y_inv - preds_inv) ** 2)
            ss_tot = np.sum((test_y_inv - np.mean(test_y_inv)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(test_y_inv)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1) if n > n_predictors + 1 else 0
            adj_r2_history.append(adj_r2)
            if adj_r2 > best_adj_r2:
                best_adj_r2 = adj_r2
                best_model = tf.keras.models.clone_model(model) if model_name not in ['xgboost', 'lightgbm'] else model
                if model_name not in ['xgboost', 'lightgbm']:
                    best_model.set_weights(model.get_weights())
        except Exception as e:
            print(f"Error in walk-forward validation for {model_name} at step {i}: {e}")
            continue
    return {'adj_r2': best_adj_r2, 'best_model': best_model}, adj_r2_history


# Step 7
stock_results = {}
results = {}
for model_name, model_fn, data_type in models:
    if not (TF_AVAILABLE and data_type == 'deep') and not (LGB_AVAILABLE and model_name == 'lightgbm') and not (XGB_AVAILABLE and model_name == 'xgboost'):
        continue
    print(f"\nTraining {model_name}...")
    X_data = stock_X_deep if data_type == 'deep' else stock_X_ml
    try:
        if OPTUNA_AVAILABLE:
            optuna.logging.set_verbosity(optuna.logging.WARNING)
            study = optuna.create_study(direction='minimize', sampler=TPESampler())
            def objective(trial):
                model = model_fn(trial)
                n_samples = len(X_data)
                train_size = int(0.8 * n_samples)
                train_X, train_y = X_data[:train_size], stock_y[:train_size]
                val_X, val_y = X_data[train_size:], stock_y[train_size:]
                if model_name in ['xgboost', 'lightgbm']:
                    model.fit(train_X.reshape(train_X.shape[0], -1), train_y)
                    preds = model.predict(val_X.reshape(val_X.shape[0], -1))
                else:
                    model.fit(train_X, train_y, epochs=5, batch_size=64, validation_split=0.2, verbose=0,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                    preds = model.predict(val_X, verbose=0).flatten()
                val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                ss_res = np.sum((val_y_inv - preds_inv) ** 2)
                ss_tot = np.sum((val_y_inv - np.mean(val_y_inv)) ** 2)
                r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                n = len(val_y_inv)
                adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                return -adj_r2
            study.optimize(objective, n_trials=3)
            model = model_fn(optuna.trial.FixedTrial(study.best_params))
        else:
            model = model_fn({})
        result, adj_r2_history = walk_forward_validation(model, model_name, X_data, stock_y, price_scaler)
        results[model_name] = result
        print(f"{model_name} - Best Adj R²: {result['adj_r2']:.6f}")
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        results[model_name] = {'adj_r2': 0, 'best_model': None}
    # NEW: release GPU memory between models only — not between walk-forward
    # steps within a model, since the model must persist across those steps.
    if model_name not in ['xgboost', 'lightgbm']:
        tf.keras.backend.clear_session()

stock_results = results


Step 6: Training and evaluating models...

Training cnn_lstm...
cnn_lstm - Best Adj R²: 0.843558

Training conv1d_lstm...
conv1d_lstm - Best Adj R²: 0.917398

Training bilstm...
bilstm - Best Adj R²: 0.961749

Training stacked_lstm...
stacked_lstm - Best Adj R²: 0.812456

Training gru...
gru - Best Adj R²: 0.967961

Training lightgbm...
[LightGBM] [Info] Total Bins 9744
[LightGBM] [Info] Number of data points in the train set: 1536, number of used features: 210
[LightGBM] [Info] Start training from score 0.420720
[LightGBM] [Info] Total Bins 9744
[LightGBM] [Info] Number of data points in the train set: 1536, number of used features: 210
[LightGBM] [Info] Start training from score 0.420720
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [28]:
# Step 6.5: Analyze Overfitting
print("\nStep 6.5: Analyzing overfitting...")

X_data = stock_X_deep if 'cnn_lstm' in stock_results else stock_X_ml
y_data = stock_y
n_samples = len(X_data)
train_size = int(0.8 * n_samples)
train_X, val_X = X_data[:train_size], X_data[train_size:]
train_y, val_y = y_data[:train_size], y_data[train_size:]

for model_name, result in stock_results.items():
    best_model = result['best_model']
    if best_model is None:
        continue
    try:
        if model_name in ['xgboost', 'lightgbm']:
            # FIX: was train_X[:, :, 0].reshape(-1, SEQ_LENGTH) — only used
            # feature 0 (price), dropping the sentiment columns.
            train_pred = best_model.predict(train_X.reshape(train_X.shape[0], -1))
            val_pred = best_model.predict(val_X.reshape(val_X.shape[0], -1))
        else:
            best_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
            best_model.fit(train_X, train_y, epochs=5, batch_size=64, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
            train_pred = best_model.predict(train_X, verbose=0).flatten()
            val_pred = best_model.predict(val_X, verbose=0).flatten()

        train_y_inv = price_scaler.inverse_transform(train_y.reshape(-1, 1)).flatten()
        val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
        train_pred_inv = price_scaler.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        val_pred_inv = price_scaler.inverse_transform(val_pred.reshape(-1, 1)).flatten()

        def calculate_adj_r2(y_true, y_pred):
            ss_res = np.sum((y_true - y_pred) ** 2)
            ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(y_true)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
            return adj_r2

        train_adj_r2 = calculate_adj_r2(train_y_inv, train_pred_inv)
        val_adj_r2 = calculate_adj_r2(val_y_inv, val_pred_inv)
        overfitting_gap = train_adj_r2 - val_adj_r2

        stock_results[model_name]['val_adj_r2'] = val_adj_r2
        stock_results[model_name]['overfitting_gap'] = overfitting_gap
        print(f"Model: {model_name} - Val Adj R²: {val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap:.6f}")
        print(f"  Training Adj R²: {train_adj_r2:.6f}")
        if overfitting_gap > 0.2:
            print("  Warning: Potential overfitting detected")
        elif overfitting_gap > 0.1:
            print("  Caution: Moderate overfitting risk")
        else:
            print("  Status: Low overfitting risk")
    except Exception as e:
        print(f"Error analyzing overfitting for {model_name}: {e}")


Step 6.5: Analyzing overfitting...
Model: cnn_lstm - Val Adj R²: 0.595062, Overfitting Gap: 0.300549
  Training Adj R²: 0.895611
Model: conv1d_lstm - Val Adj R²: 0.068499, Overfitting Gap: 0.806228
  Training Adj R²: 0.874726
Model: bilstm - Val Adj R²: 0.866450, Overfitting Gap: 0.103757
  Training Adj R²: 0.970207
  Caution: Moderate overfitting risk
Model: stacked_lstm - Val Adj R²: 0.852030, Overfitting Gap: 0.082391
  Training Adj R²: 0.934421
  Status: Low overfitting risk
Model: gru - Val Adj R²: 0.906055, Overfitting Gap: 0.068426
  Training Adj R²: 0.974481
  Status: Low overfitting risk
Model: lightgbm - Val Adj R²: 0.994127, Overfitting Gap: 0.005792
  Training Adj R²: 0.999919
  Status: Low overfitting risk
Model: xgboost - Val Adj R²: 0.992653, Overfitting Gap: 0.006146
  Training Adj R²: 0.998799
  Status: Low overfitting risk
Model: transformer - Val Adj R²: 0.627085, Overfitting Gap: 0.280188
  Training Adj R²: 0.907273
Model: tcn - Val Adj R²: 0.729795, Overfitting Ga

In [29]:
# Step 7: Generate Predictions with Best Non-Overfit Model
print("\nStep 7: Generating predictions...")
best_val_adj_r2 = -float('inf')
best_model_name = None
best_model = None

for model_name, result in stock_results.items():
    if result.get('val_adj_r2', -float('inf')) > best_val_adj_r2 and result.get('overfitting_gap', float('inf')) <= 0.2:
        best_val_adj_r2 = result['val_adj_r2']
        best_model_name = model_name
        best_model = result['best_model']

if best_model_name is None:
    print("No model with acceptable overfitting gap found. Using best overall Adj R².")
    best_model_name = max(stock_results.items(), key=lambda x: x[1]['adj_r2'])[0]
    best_model = stock_results[best_model_name]['best_model']

overfitting_gap = stock_results[best_model_name].get('overfitting_gap', 'N/A')
print(f"Best model: {best_model_name} with Val Adj R²: {best_val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap}")

def generate_predictions(model, model_name, scaled_data, price_scaler, timestamps, steps=150):
    preds_scaled = []
    # FIX: was scaled_data[:SEQ_LENGTH] — the OLDEST 30 rows of the 300-row
    # window, meaning the forecast started ~270 minutes before the actual
    # end of the data. Use the most recent 30 rows instead.
    current_sequence = scaled_data[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, feature_dim)
    for _ in range(steps):
        try:
            if model_name in ['xgboost', 'lightgbm']:
                # FIX: was current_sequence[:, :, 0].reshape(1, -1) — only
                # used feature 0 (price), dropping sentiment.
                pred_scaled = model.predict(current_sequence.reshape(1, -1))[0]
            else:
                pred_scaled = model.predict(current_sequence, verbose=0)[0, 0]
            preds_scaled.append(float(pred_scaled))
            current_sequence = np.roll(current_sequence, -1, axis=1)
            current_sequence[0, -1, 0] = pred_scaled
            if feature_dim > 1:
                current_sequence[0, -1, 1:] = current_sequence[0, -2, 1:]
        except Exception as e:
            print(f"Prediction failed for {model_name}: {e}")
            break
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    predictions = price_scaler.inverse_transform(preds_scaled).flatten()
    last_timestamp = pd.to_datetime(timestamps.iloc[-1])
    future_timestamps = pd.date_range(start=last_timestamp + pd.Timedelta(minutes=1), periods=steps, freq='min')
    return predictions, future_timestamps

predictions, future_timestamps = generate_predictions(best_model, best_model_name, last_300_scaled, price_scaler, stock_data['Timestamp'])
print(f"Predictions (first 5): {predictions[:5]}")
print(f"Future timestamps (first 5): {future_timestamps[:5]}")


Step 7: Generating predictions...
Best model: lightgbm with Val Adj R²: 0.994127, Overfitting Gap: 0.005792183019699526
Predictions (first 5): [335.73283876 335.65315588 335.60474459 335.63386993 335.65756509]
Future timestamps (first 5): DatetimeIndex(['2026-09-18 20:00:00+00:00', '2026-09-18 20:01:00+00:00',
               '2026-09-18 20:02:00+00:00', '2026-09-18 20:03:00+00:00',
               '2026-09-18 20:04:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='min')


In [31]:
# Step 8: Generate Predictions for All Valid Models
print("\nStep 8: Generating predictions for all models with Adj R² > 0.8...")
valid_models = {}
for model_name, result in stock_results.items():
    if result.get('val_adj_r2', 0) > 0.8 and result.get('overfitting_gap', float('inf')) <= 0.2:
        valid_models[model_name] = result['best_model']
        print(f"Model '{model_name}' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)")

predictions_df = pd.DataFrame()
if not valid_models:
    print("No valid models found.")
else:
    for model_name, model in valid_models.items():
        print(f"\nGenerating predictions for {model_name}...")
        predictions, future_timestamps = generate_predictions(model, model_name, last_300_scaled, price_scaler, stock_data['Timestamp'])
        predictions_df[model_name] = predictions
    predictions_df['Timestamp'] = future_timestamps

print("\nPredictions:")
print(predictions_df.head())
filename = f'{TARGET_COLUMN}_predictions.csv'
predictions_df.to_csv(filename, index=False)
print(f"\nDownloading {filename}...")
files.download(filename)


Step 8: Generating predictions for all models with Adj R² > 0.8...
Model 'bilstm' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'stacked_lstm' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'gru' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'lightgbm' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'xgboost' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)

Generating predictions for bilstm...

Generating predictions for stacked_lstm...

Generating predictions for gru...

Generating predictions for lightgbm...

Generating predictions for xgboost...

Predictions:
       bilstm  stacked_lstm         gru    lightgbm     xgboost  \
0  336.465959    336.039509  335.751338  335.732839  335.742733   
1  336.420570    336.065620  335.652983  335.653156  335.697537   
2  336.382667    336.081142  335.555318  335.604745  335.665637   
3  336.348837    336.085398  335.460131  335.633870  335.604165   
4  336.322927   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>